# Data exploration

Initial exploration 

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
sensors = pd.read_csv("../data/fyta_sensor_sample.csv")
context = pd.read_csv("../data/fyta_contextual.csv")
images = pd.read_csv("../data/fyta_images.csv")

context["created_at"] = pd.to_datetime(context["created_at"])
images["captured_at"] = pd.to_datetime(images["captured_at"])

print(sensors.shape)
print(context.shape)
print(images.shape)

(15950, 8)
(49, 5)
(28, 5)


In [3]:
display(sensors.head())
display(sensors.describe(include="all"))
display(sensors.isna().sum())
display(sensors.dtypes)

,device_id,timestamp,substrate_label,soil_moisture_vwc,soil_temp_c,ec_us_cm,light_par,air_humidity_pct
0,SENS-01,2026-06-01 00:00:00,potting_soil,40.93,18.65,733.0,0.0,59.6
1,SENS-01,2026-06-01 00:15:00,potting_soil,40.68,17.95,694.0,0.0,58.8
2,SENS-01,2026-06-01 00:30:00,potting_soil,40.63,18.52,671.0,0.0,63.6
3,SENS-01,2026-06-01 00:45:00,potting_soil,40.29,18.47,684.0,0.0,64.4
4,SENS-01,2026-06-01 01:00:00,potting_soil,40.29,17.19,652.0,0.0,64.0


,device_id,timestamp,substrate_label,soil_moisture_vwc,soil_temp_c,ec_us_cm,light_par,air_humidity_pct
count,15950,15950,15950,15950.000000,15950.000000,15950.000000,15950.000000,15950.000000
unique,8,4032,3,NaN,NaN,NaN,NaN,NaN
top,SENS-08,2026-06-02 03:15:00,potting_soil,NaN,NaN,NaN,NaN,NaN
freq,2031,8,12111,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,29.359522,27.161629,749.939185,102.528489,55.046464
std,NaN,NaN,NaN,12.551057,16.464319,220.635514,123.467364,13.176824
min,NaN,NaN,NaN,-5.430000,16.100000,556.000000,0.000000,27.500000
25%,NaN,NaN,NaN,22.860000,18.850000,675.000000,0.000000,42.700000
50%,NaN,NaN,NaN,26.050000,21.770000,703.000000,11.100000,55.100000
75%,NaN,NaN,NaN,33.560000,24.100000,735.000000,226.700000,67.300000


device_id            0
timestamp            0
substrate_label      0
soil_moisture_vwc    0
soil_temp_c          0
ec_us_cm             0
light_par            0
air_humidity_pct     0
dtype: int64

device_id                str
timestamp                str
substrate_label          str
soil_moisture_vwc    float64
soil_temp_c          float64
ec_us_cm             float64
light_par            float64
air_humidity_pct     float64
dtype: object

## Expected sampling behaviour
There are 8 sensors and approximately 3 weeks.
At 15-minute intervals: 2,016 expected readings/device.

In [4]:
sensors.groupby("device_id").size()

device_id
SENS-01    2016
SENS-02    2016
SENS-03    2016
SENS-04    2016
SENS-05    1823
SENS-06    2016
SENS-07    2016
SENS-08    2031
dtype: int64

### Problem: SENS-05 - missing data
It has 1,823 rather than 2,016 readings.

Calculate gaps explicitly
Flag this as missing_data = True


### Problem: SENS-08 - duplicate measurements
SENS-08 has 2,031 records rather than 2,016.
flag this as duplicate_record = True

## Format, range and physical plausibility checks

Review by column

### Timestamp

### Problem: Format isn't completely consistent.
Present 'YYYY-MM-DD' and 'DD/MM/YYYY'

In [5]:
sensors.groupby("device_id")["timestamp"].describe()

,count,unique,top,freq
device_id,,,,
SENS-01,2016,2016,2026-06-01 00:00:00,1
SENS-02,2016,2016,2026-06-01 00:00:00,1
SENS-03,2016,2016,2026-06-01 00:00:00,1
SENS-04,2016,2016,2026-06-01 00:00:00,1
SENS-05,1823,1823,2026-06-01 00:00:00,1
SENS-06,2016,2016,2026-06-01 00:00:00,1
SENS-07,2016,2016,01/06/2026 02:00,1
SENS-08,2031,2016,2026-06-02 03:15:00,2


### substrate_label
Validate entries from app, could be: potting_soil, coco_coir, orchid_bark

Status: Ok

In [6]:
sensors.groupby("device_id")["substrate_label"].unique()

device_id
SENS-01    [potting_soil]
SENS-02    [potting_soil]
SENS-03    [potting_soil]
SENS-04    [potting_soil]
SENS-05       [coco_coir]
SENS-06    [potting_soil]
SENS-07     [orchid_bark]
SENS-08    [potting_soil]
Name: substrate_label, dtype: object

### soil_moisture_vwc

Volumetric water content, % (typically ~0–60% in real substrates)

### Problem: impossible moisture values
SENS-4 presents over 100% VWC values. Max=107.12 with relatively bigger std = 23.532003. Review sensor?

SENS-08 presents negative VWC values. VWC cannot physically be negative. Min = -5.43

In [7]:
sensors.groupby("device_id")["soil_moisture_vwc"].describe()

,count,mean,std,min,25%,50%,75%,max
device_id,,,,,,,,
SENS-01,2016.0,28.161870,5.253081,18.12,24.6775,26.410,30.8400,43.67
SENS-02,2016.0,33.515382,6.854919,20.70,29.3725,33.330,38.2000,51.52
SENS-03,2016.0,33.924033,6.303578,23.56,28.7975,33.665,37.3000,51.12
SENS-04,2016.0,45.337391,23.532003,15.07,25.9950,37.495,60.0325,107.12
SENS-05,1823.0,23.580340,5.873940,16.16,18.5950,21.170,27.5600,38.77
SENS-06,2016.0,26.005491,3.104600,20.59,25.0100,25.010,25.0100,40.13
SENS-07,2016.0,17.631855,4.962536,9.36,13.6375,16.265,20.7000,31.44
SENS-08,2031.0,26.190133,5.378476,-5.43,22.6000,25.250,29.2500,39.59


### soil_temp_c

Soil temperature, °C

### Problem: SENS-07 has suspicious values for soil temperature
soil_temp_c min = 61.86
soil_temp_c max = 78.01

and all 2,016 observations are above 60°C since min = 61.86.
That's obviously suspicious for soil temperature.

Transform to °F 

In [8]:
sensors.groupby("device_id")["soil_temp_c"].describe()

,count,mean,std,min,25%,50%,75%,max
device_id,,,,,,,,
SENS-01,2016.0,20.978566,2.507553,16.56,18.5175,21.010,23.4025,25.54
SENS-02,2016.0,20.992743,2.516909,16.42,18.5200,20.995,23.4425,25.97
SENS-03,2016.0,20.994980,2.497716,16.49,18.5700,21.020,23.4500,25.60
SENS-04,2016.0,21.002371,2.515776,16.51,18.5400,20.930,23.4600,25.63
SENS-05,1823.0,20.995590,2.514888,16.10,18.5900,21.020,23.4300,25.40
SENS-06,2016.0,21.003676,2.497465,16.24,18.6075,20.970,23.4225,25.64
SENS-07,2016.0,69.793889,4.528584,61.86,65.4100,69.755,74.1200,78.01
SENS-08,2031.0,20.986864,2.525953,16.43,18.4800,20.960,23.4400,25.43


### ec_us_cm

Electrical conductivity / salinity, µS/cm

### Problem: SENS-03 - sensor-quality problem.
min = 561.0, Max=2266.0 mean = 1093.937500, ask domain expert



In [9]:
sensors.groupby("device_id")["ec_us_cm"].describe()

,count,mean,std,min,25%,50%,75%,max
device_id,,,,,,,,
SENS-01,2016.0,699.632937,40.004493,577.0,674.0,698.0,727.00,832.0
SENS-02,2016.0,702.834821,40.106088,570.0,677.0,703.0,729.00,828.0
SENS-03,2016.0,1093.937500,488.413807,561.0,705.0,794.5,1483.75,2266.0
SENS-04,2016.0,699.110119,40.354727,556.0,672.0,699.0,726.00,832.0
SENS-05,1823.0,698.827208,40.044340,561.0,671.0,697.0,726.00,831.0
SENS-06,2016.0,700.584325,41.074233,563.0,673.0,700.0,728.00,867.0
SENS-07,2016.0,701.502976,40.480244,579.0,675.0,701.0,729.00,845.0
SENS-08,2031.0,698.572624,39.980100,562.0,671.0,699.0,725.00,840.0


### light_par
Light intensity (PAR)

### Problem SENS-08 high light readings
max = 589
high compared to other sensores, ask domain expert

validate timing with logs

In [10]:
sensors.groupby("device_id")["light_par"].describe()

,count,mean,std,min,25%,50%,75%,max
device_id,,,,,,,,
SENS-01,2016.0,102.209474,123.308258,0.0,0.0,10.30,226.400,339.9
SENS-02,2016.0,102.303125,123.311093,0.0,0.0,9.65,223.250,335.3
SENS-03,2016.0,102.595188,123.305808,0.0,0.0,11.45,227.600,340.2
SENS-04,2016.0,102.423165,123.461503,0.0,0.0,10.50,226.700,345.8
SENS-05,1823.0,102.202304,123.109012,0.0,0.0,10.00,226.650,335.0
SENS-06,2016.0,102.229563,123.061455,0.0,0.0,11.25,226.000,338.0
SENS-07,2016.0,102.326488,123.168902,0.0,0.0,11.95,227.375,331.5
SENS-08,2031.0,103.897194,125.157725,0.0,0.0,11.80,227.600,589.0


### air_humidity_pct
Relative air humidity, %

### Problem: SENS-08 humidity >100%

values up to 131%. physically impossible as relative humidity.



In [11]:
sensors.groupby("device_id")["air_humidity_pct"].describe()

,count,mean,std,min,25%,50%,75%,max
device_id,,,,,,,,
SENS-01,2016.0,55.050744,13.055255,27.8,42.6,55.00,67.3,80.0
SENS-02,2016.0,54.990228,13.184000,29.7,42.7,55.50,67.4,81.7
SENS-03,2016.0,55.091319,13.104645,29.6,42.8,55.35,67.3,81.2
SENS-04,2016.0,54.987698,12.979983,29.2,42.8,55.05,67.3,79.4
SENS-05,1823.0,54.967581,13.053513,29.1,42.7,55.10,67.2,87.9
SENS-06,2016.0,54.946528,13.110249,29.8,42.7,55.10,67.3,81.5
SENS-07,2016.0,55.084921,13.140753,28.2,42.6,55.25,67.3,80.8
SENS-08,2031.0,55.243673,13.773774,27.5,42.5,55.10,67.4,131.0
